# Breast Ultrasound Image Classification


In [1]:
import kagglehub
import shutil
import os

path = kagglehub.dataset_download("sabahesaraki/breast-ultrasound-images-dataset")

dest = os.path.join(os.getcwd(), "breast_ultrasound_dataset")

if not os.path.exists(dest):
    shutil.move(path, dest)
    print("Dataset saved in:", dest)
else:
    print("Dataset already exists at:", dest)

c:\Users\Asus\Desktop\Deep_learning_medical_images\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset already exists at: c:\Users\Asus\Desktop\Deep_learning_medical_images\tumor_classification(BUSI)\breast_ultrasound_dataset


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from collections import Counter
import numpy as np
from PIL import Image
import random
import matplotlib
matplotlib.use('Agg')  # non-interactive backend, safe for all environments
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print("All imports successful.")

All imports successful.


In [3]:
SEED = 42

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

g = torch.Generator()
g.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [4]:
DATA_DIR = os.path.join(
    os.getcwd(),
    "breast_ultrasound_dataset",
    "Dataset_BUSI_with_GT"
)

def is_valid_file(path):
    return "_mask" not in path

dataset = ImageFolder(DATA_DIR, transform=None, is_valid_file=is_valid_file)

labels = [label for _, label in dataset.samples]
CLASS_NAMES = list(dataset.class_to_idx.keys())
print("Class mapping:", dataset.class_to_idx)
print("Dataset distribution:", Counter(labels))

Class mapping: {'benign': 0, 'malignant': 1, 'normal': 2}
Dataset distribution: Counter({0: 437, 1: 210, 2: 133})


In [5]:
train_idx, test_idx = train_test_split(
    range(len(labels)),
    test_size=0.2,
    stratify=labels,
    random_state=SEED
)

print(f"Train samples: {len(train_idx)}  |  Test samples: {len(test_idx)}")

Train samples: 624  |  Test samples: 156


In [6]:
augment_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x + 0.05 * torch.randn_like(x))
])

normal_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

print("Transforms defined.")

Transforms defined.


In [7]:
class AugmentedDataset(Dataset):

    def __init__(self, dataset, indices, labels):
        self.dataset = dataset
        self.indices = indices
        self.labels = labels
        self.extra_samples = []

        malignant = 1
        normal = 2

        malignant_idx = [i for i in indices if labels[i] == malignant]
        normal_idx    = [i for i in indices if labels[i] == normal]

        malignant_extra = int(len(malignant_idx) * 1.0)
        normal_extra    = int(len(normal_idx) * 1.0)

        random.seed(SEED)
        self.extra_samples += random.choices(malignant_idx, k=malignant_extra)
        self.extra_samples += random.choices(normal_idx,    k=normal_extra)

        print("Extra augmented samples:", len(self.extra_samples))

    def __len__(self):
        return len(self.indices) + len(self.extra_samples)

    def __getitem__(self, idx):
        if idx < len(self.indices):
            real_idx = self.indices[idx]
            path, label = self.dataset.samples[real_idx]
            img = Image.open(path).convert("RGB")
            return normal_transform(img), label
        else:
            aug_idx = self.extra_samples[idx - len(self.indices)]
            path, label = self.dataset.samples[aug_idx]
            img = Image.open(path).convert("RGB")
            return augment_transform(img), label


class TestDataset(Dataset):

    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        path, label = self.dataset.samples[real_idx]
        img = Image.open(path).convert("RGB")
        return normal_transform(img), label


print("Dataset classes defined.")

Dataset classes defined.


In [8]:
train_dataset = AugmentedDataset(dataset, train_idx, labels)
test_dataset  = TestDataset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  generator=g)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, generator=g)

print(f"Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}")

Extra augmented samples: 274
Train batches: 29  |  Test batches: 5


In [9]:
class DepthwiseSeparableConv(nn.Module):
    """Depthwise conv (spatial) followed by pointwise conv (channel mixing)."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.depthwise  = nn.Conv2d(in_ch, in_ch,  kernel_size=3, stride=stride,
                                    padding=1, groups=in_ch, bias=False)  # spatial filter per channel
        self.pointwise  = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)  # channel projection
        self.bn         = nn.BatchNorm2d(out_ch)
        self.relu       = nn.ReLU()

    def forward(self, x):
        return self.relu(self.bn(self.pointwise(self.depthwise(x))))


class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        # First block uses a standard conv to handle the RGB → feature-map transition
        # (depthwise on 3-channel input wastes capacity since groups=3 is very small)
        self.features = nn.Sequential(
            # Block 1 — standard conv entry
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2 — depthwise separable
            DepthwiseSeparableConv(32, 64),
            nn.MaxPool2d(2),

            # Block 3 — depthwise separable
            DepthwiseSeparableConv(64, 128),
            nn.MaxPool2d(2),

            # Block 4 — depthwise separable
            DepthwiseSeparableConv(128, 256),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = CNN().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

CNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): DepthwiseSeparableConv(
      (depthwise): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
      (pointwise): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
    )
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): DepthwiseSeparableConv(
      (depthwise): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
      (pointwise): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(128, eps=1e-05, moment

In [10]:
class FocalLoss(nn.Module):

    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(inputs, targets, reduction='none')
        pt         = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.alpha is not None:
            focal_loss = self.alpha[targets] * focal_loss
        return focal_loss.mean()


alpha     = torch.tensor([1.0, 1.2, 2.8]).to(device)
criterion = FocalLoss(alpha=alpha, gamma=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

print("Loss function and optimizer ready.")

Loss function and optimizer ready.


In [11]:
EPOCHS = 30
BEST_MODEL_PATH = "best_model.pth"

history = {"epoch": [], "train_loss": [], "train_acc": [], "val_loss": []}

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    # ── training ──────────────────────────────────────────────────────────
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels_batch in train_loader:
        images       = images.to(device)
        labels_batch = labels_batch.to(device)

        outputs = model(images)
        loss    = criterion(outputs, labels_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds       = torch.argmax(outputs, 1)
        correct    += (preds == labels_batch).sum().item()
        total      += labels_batch.size(0)

    epoch_loss = total_loss / len(train_loader)
    epoch_acc  = correct / total

    # ── validation loss ───────────────────────────────────────────────────
    model.eval()
    val_loss_total = 0
    with torch.no_grad():
        for images, labels_batch in test_loader:
            images       = images.to(device)
            labels_batch = labels_batch.to(device)
            outputs      = model(images)
            val_loss_total += criterion(outputs, labels_batch).item()
    val_loss = val_loss_total / len(test_loader)
    scheduler.step(val_loss) 

    # ── checkpoint on best val loss ───────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        ckpt_flag = "  ✓ saved"
    else:
        ckpt_flag = ""

    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_loss)
    history["train_acc"].append(epoch_acc)
    history["val_loss"].append(val_loss)

    print(f"Epoch {epoch:02d} | Loss {epoch_loss:.4f} | Train Acc {epoch_acc:.4f} "
          f"| Val Loss {val_loss:.4f}{ckpt_flag}")

# ── load best weights back into model ─────────────────────────────────────
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
print(f"Best model loaded from: {BEST_MODEL_PATH}")

Epoch 00 | Loss 0.7037 | Train Acc 0.4621 | Val Loss 0.6720  ✓ saved
Epoch 01 | Loss 0.6212 | Train Acc 0.3497 | Val Loss 0.6486  ✓ saved
Epoch 02 | Loss 0.5847 | Train Acc 0.4287 | Val Loss 0.6366  ✓ saved
Epoch 03 | Loss 0.5861 | Train Acc 0.4755 | Val Loss 0.6304  ✓ saved
Epoch 04 | Loss 0.5671 | Train Acc 0.4833 | Val Loss 0.6232  ✓ saved
Epoch 05 | Loss 0.5666 | Train Acc 0.4599 | Val Loss 0.6162  ✓ saved
Epoch 06 | Loss 0.5437 | Train Acc 0.5434 | Val Loss 0.6121  ✓ saved
Epoch 07 | Loss 0.5534 | Train Acc 0.4644 | Val Loss 0.6033  ✓ saved
Epoch 08 | Loss 0.5319 | Train Acc 0.5401 | Val Loss 0.5973  ✓ saved
Epoch 09 | Loss 0.5219 | Train Acc 0.4688 | Val Loss 0.5940  ✓ saved
Epoch 10 | Loss 0.5027 | Train Acc 0.5490 | Val Loss 0.5872  ✓ saved
Epoch 11 | Loss 0.5021 | Train Acc 0.5646 | Val Loss 0.5919
Epoch 12 | Loss 0.4946 | Train Acc 0.5924 | Val Loss 0.5858  ✓ saved
Epoch 13 | Loss 0.4911 | Train Acc 0.5234 | Val Loss 0.5884
Epoch 14 | Loss 0.4884 | Train Acc 0.5802 | Val Loss

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = history["epoch"]

axes[0].plot(ep, history["train_loss"], color="#378ADD", linewidth=2,
             marker="o", markersize=4, label="Train loss")
axes[0].plot(ep, history["val_loss"],   color="#D85A30", linewidth=2,
             marker="s", markersize=4, label="Val loss", linestyle="--")
axes[0].set_title("Training vs Validation loss", fontsize=13)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Focal loss")
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history["train_acc"], color="#1D9E75", linewidth=2,
             marker="o", markersize=4, label="Train acc")
axes[1].set_title("Training accuracy", fontsize=13)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.7, 0.85)
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle("Training curves", fontsize=15, y=1.02)
fig.tight_layout()

training_curve_path = "training_curves.png"
fig.savefig(training_curve_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", training_curve_path)

Saved: training_curves.png


C:\Users\Asus\AppData\Local\Temp\ipykernel_33196\1658602934.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
model.eval()

y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for images, labels_batch in test_loader:
        images  = images.to(device)
        outputs = model(images)
        probs   = torch.softmax(outputs, dim=1)
        preds   = torch.argmax(outputs, dim=1)

        y_true.extend(labels_batch.numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
conf   = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
roc    = roc_auc_score(y_true, y_prob, multi_class="ovr")

print(report)
print("Confusion Matrix:")
print(conf)
print(f"\nROC-AUC (macro OvR): {roc:.4f}")

              precision    recall  f1-score   support

      benign       0.71      0.75      0.73        87
   malignant       0.65      0.62      0.63        42
      normal       0.50      0.44      0.47        27

    accuracy                           0.66       156
   macro avg       0.62      0.60      0.61       156
weighted avg       0.66      0.66      0.66       156

Confusion Matrix:
[[65 11 11]
 [15 26  1]
 [12  3 12]]

ROC-AUC (macro OvR): 0.7300


In [14]:
y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
colors = ["#378ADD", "#D85A30", "#1D9E75"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc     = auc(fpr, tpr)

    axes[i].plot(fpr, tpr, color=color, linewidth=2, label=f"AUC = {roc_auc:.3f}")
    axes[i].plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
    axes[i].fill_between(fpr, tpr, alpha=0.08, color=color)
    axes[i].set_title(f"ROC — {cls_name}", fontsize=13)
    axes[i].set_xlabel("False positive rate")
    axes[i].set_ylabel("True positive rate")
    axes[i].set_xlim(0, 1)
    axes[i].set_ylim(0, 1.05)
    axes[i].legend(loc="lower right")
    axes[i].grid(True, alpha=0.3)

fig.suptitle(f"ROC curves  (macro OvR AUC = {roc:.4f})", fontsize=15, y=1.02)
fig.tight_layout()

roc_curve_path = "roc_curves.png"
fig.savefig(roc_curve_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", roc_curve_path)

Saved: roc_curves.png


C:\Users\Asus\AppData\Local\Temp\ipykernel_33196\1430892867.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
fig, ax = plt.subplots(figsize=(6, 5))

im = ax.imshow(conf, interpolation="nearest", cmap="Blues")
fig.colorbar(im, ax=ax)

tick_marks = np.arange(len(CLASS_NAMES))
ax.set_xticks(tick_marks)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.set_yticks(tick_marks)
ax.set_yticklabels(CLASS_NAMES)

thresh = conf.max() / 2.0
for i in range(conf.shape[0]):
    for j in range(conf.shape[1]):
        ax.text(j, i, str(conf[i, j]),
                ha="center", va="center", fontsize=12,
                color="white" if conf[i, j] > thresh else "black")

ax.set_title("Confusion matrix", fontsize=13)
ax.set_ylabel("True label")
ax.set_xlabel("Predicted label")
fig.tight_layout()

conf_matrix_path = "confusion_matrix.png"
fig.savefig(conf_matrix_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", conf_matrix_path)

Saved: confusion_matrix.png


C:\Users\Asus\AppData\Local\Temp\ipykernel_33196\1677011072.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
output_file = "focal_normal_results4.txt"

with open(output_file, "w") as f:
    f.write("Classification Report\n")
    f.write(report)
    f.write("\n\nConfusion Matrix\n")
    f.write(str(conf))
    f.write("\n\nROC-AUC\n")
    f.write(str(roc))
    f.write("\n\nSaved plots\n")
    f.write(f"  Training curves : {training_curve_path}\n")
    f.write(f"  ROC curves      : {roc_curve_path}\n")
    f.write(f"  Confusion matrix: {conf_matrix_path}\n")

print("Results saved to:", output_file)
print("\nAll output files:")
for f in [output_file, training_curve_path, roc_curve_path, conf_matrix_path]:
    print(" ", f)

Results saved to: focal_normal_results4.txt

All output files:
  focal_normal_results4.txt
  training_curves.png
  roc_curves.png
  confusion_matrix.png
